In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time

# Data Reading

In [3]:
df = spark.read.format("delta")\
    .load("s3://learn-databricks-project-e2e-1-bronze/customers")
    
df.show()

+-----------+----------+---------+--------------------+------------------+-----+-------------+
|customer_id|first_name|last_name|               email|              city|state|_rescued_data|
+-----------+----------+---------+--------------------+------------------+-----+-------------+
|     C00001|     Emily|   Mooney|   rushjeff@ryan.org|      Johnsonmouth|   MS|         NULL|
|     C00002|    Andrea|  Sellers|mccoykiara@kelly.com|       Stephenfort|   WY|         NULL|
|     C00003|     Craig|    Hayes|rebeccamiller@yah...|South Stephenshire|   LA|         NULL|
|     C00004|     Bryan|    Scott|lawrence05@campbe...|         Chrisland|   ND|         NULL|
|     C00005|      Sean|  Vasquez|  carrie45@yahoo.com|   East Dennistown|   RI|         NULL|
|     C00006|     Kevin| Mccarthy|traceyramos@gmail...|     North Matthew|   IN|         NULL|
|     C00007|    Amanda|    Doyle|scottallen@gmail.com|        Joneshaven|   VA|         NULL|
|     C00008|      Paul|   Campos|sullivanjeremy@h

In [4]:
df = df.drop("_rescued_data")
df.show()

+-----------+----------+---------+--------------------+------------------+-----+
|customer_id|first_name|last_name|               email|              city|state|
+-----------+----------+---------+--------------------+------------------+-----+
|     C00001|     Emily|   Mooney|   rushjeff@ryan.org|      Johnsonmouth|   MS|
|     C00002|    Andrea|  Sellers|mccoykiara@kelly.com|       Stephenfort|   WY|
|     C00003|     Craig|    Hayes|rebeccamiller@yah...|South Stephenshire|   LA|
|     C00004|     Bryan|    Scott|lawrence05@campbe...|         Chrisland|   ND|
|     C00005|      Sean|  Vasquez|  carrie45@yahoo.com|   East Dennistown|   RI|
|     C00006|     Kevin| Mccarthy|traceyramos@gmail...|     North Matthew|   IN|
|     C00007|    Amanda|    Doyle|scottallen@gmail.com|        Joneshaven|   VA|
|     C00008|      Paul|   Campos|sullivanjeremy@ho...|  South Nathanfurt|   CT|
|     C00009|      Mary|    Green|  dennis03@yahoo.com|      Kimberlyview|   MD|
|     C00010|     James|    

In [5]:
df = df.withColumn("domains", split(col("email"), "@")[1])
df.show()

+-----------+----------+---------+--------------------+------------------+-----+------------------+
|customer_id|first_name|last_name|               email|              city|state|           domains|
+-----------+----------+---------+--------------------+------------------+-----+------------------+
|     C00001|     Emily|   Mooney|   rushjeff@ryan.org|      Johnsonmouth|   MS|          ryan.org|
|     C00002|    Andrea|  Sellers|mccoykiara@kelly.com|       Stephenfort|   WY|         kelly.com|
|     C00003|     Craig|    Hayes|rebeccamiller@yah...|South Stephenshire|   LA|         yahoo.com|
|     C00004|     Bryan|    Scott|lawrence05@campbe...|         Chrisland|   ND|     campbell.info|
|     C00005|      Sean|  Vasquez|  carrie45@yahoo.com|   East Dennistown|   RI|         yahoo.com|
|     C00006|     Kevin| Mccarthy|traceyramos@gmail...|     North Matthew|   IN|         gmail.com|
|     C00007|    Amanda|    Doyle|scottallen@gmail.com|        Joneshaven|   VA|         gmail.com|


In [6]:
df2 = df.groupBy("domains").agg(count("customer_id").alias("total_customers"))\
    .sort("total_customers", ascending=False)
df2.show()

+-------------+---------------+
|      domains|total_customers|
+-------------+---------------+
|    gmail.com|            374|
|  hotmail.com|            360|
|    yahoo.com|            331|
|    davis.com|              8|
|    brown.com|              8|
|    smith.com|              7|
|hernandez.com|              5|
|  johnson.com|              5|
|  kennedy.com|              4|
|    white.com|              4|
|   miller.com|              4|
| martinez.com|              3|
|    berry.com|              3|
|    allen.com|              3|
|    olson.com|              3|
|    davis.org|              3|
|   brown.info|              3|
|  jackson.com|              3|
|rodriguez.com|              3|
|    perez.com|              3|
+-------------+---------------+
only showing top 20 rows


In [7]:
df_gmail = df.filter(col("domains") == "gmail.com")
df_gmail.show()
time.sleep(5)  # Simulate a delay for demonstration purposes

df_yahoo = df.filter(col("domains") == "yahoo.com")
df_yahoo.show()
time.sleep(5) 

df_hotmail = df.filter(col("domains") == "hotmail.com")
df_hotmail.show()
time.sleep(5) 

+-----------+----------+----------+--------------------+-----------------+-----+---------+
|customer_id|first_name| last_name|               email|             city|state|  domains|
+-----------+----------+----------+--------------------+-----------------+-----+---------+
|     C00006|     Kevin|  Mccarthy|traceyramos@gmail...|    North Matthew|   IN|gmail.com|
|     C00007|    Amanda|     Doyle|scottallen@gmail.com|       Joneshaven|   VA|gmail.com|
|     C00011|     Jacob|        Le|   anita65@gmail.com|      Houstonfurt|   AR|gmail.com|
|     C00012|      Chad|     Banks|beardtravis@gmail...|East Jenniferview|   WI|gmail.com|
|     C00013|     James|    Martin|     jwood@gmail.com|Lake Gregoryshire|   OR|gmail.com|
|     C00017|    Angela|    Carter|rhondaferguson@gm...| West Loriborough|   GA|gmail.com|
|     C00028|     Barry|     Baker|  steven27@gmail.com|North Gregoryfurt|   NM|gmail.com|
|     C00030|     Holly|   Collins|   cfuller@gmail.com|   Port Coltonton|   AL|gmail.com|

+-----------+----------+---------+--------------------+------------------+-----+---------+
|customer_id|first_name|last_name|               email|              city|state|  domains|
+-----------+----------+---------+--------------------+------------------+-----+---------+
|     C00003|     Craig|    Hayes|rebeccamiller@yah...|South Stephenshire|   LA|yahoo.com|
|     C00005|      Sean|  Vasquez|  carrie45@yahoo.com|   East Dennistown|   RI|yahoo.com|
|     C00009|      Mary|    Green|  dennis03@yahoo.com|      Kimberlyview|   MD|yahoo.com|
|     C00014|    Thomas|  Hartman|    lmoore@yahoo.com|  East Tiffanybury|   MS|yahoo.com|
|     C00024|   Heather|    Owens|andreawilliams@ya...|       Stewartport|   NM|yahoo.com|
|     C00025| Christina|  Bennett|    dawn65@yahoo.com|    Port Juliebury|   IN|yahoo.com|
|     C00026|      Mark|   Harris| rebecca20@yahoo.com|        Ortizshire|   WY|yahoo.com|
|     C00042|     Jenna|    Lewis|hannasteven@yahoo...|  South Garrettton|   IA|yahoo.com|

+-----------+----------+---------+--------------------+-------------------+-----+-----------+
|customer_id|first_name|last_name|               email|               city|state|    domains|
+-----------+----------+---------+--------------------+-------------------+-----+-----------+
|     C00015|     Diane|   Harris|daniellowe@hotmai...|     South Courtney|   SC|hotmail.com|
|     C00020|      Juan|  Collins|rojassandra@hotma...|        South Oscar|   NV|hotmail.com|
|     C00031|  Jeanette|    Smith|kramerkaylee@hotm...|         Travisview|   CT|hotmail.com|
|     C00033|    Carrie|  Wheeler|lindasummers@hotm...|South Andrewchester|   CO|hotmail.com|
|     C00036|     Jacob|    Mccoy|jesustaylor@hotma...| Port Nicoleborough|   AR|hotmail.com|
|     C00037|     Holly|   Arnold|  eric78@hotmail.com|         Millerfurt|   VT|hotmail.com|
|     C00038|    Robert|     Haas| ythomas@hotmail.com|   Port Michaelstad|   MS|hotmail.com|
|     C00039|    Teresa|     Ward|angela55@hotmail.com|  Lak

In [8]:
df_wfn = df.withColumn("full_name", concat(col('first_name'), lit(' '), col('last_name')))\
    .drop("first_name", "last_name")
    
df_wfn.show()

+-----------+--------------------+------------------+-----+------------------+--------------+
|customer_id|               email|              city|state|           domains|     full_name|
+-----------+--------------------+------------------+-----+------------------+--------------+
|     C00001|   rushjeff@ryan.org|      Johnsonmouth|   MS|          ryan.org|  Emily Mooney|
|     C00002|mccoykiara@kelly.com|       Stephenfort|   WY|         kelly.com|Andrea Sellers|
|     C00003|rebeccamiller@yah...|South Stephenshire|   LA|         yahoo.com|   Craig Hayes|
|     C00004|lawrence05@campbe...|         Chrisland|   ND|     campbell.info|   Bryan Scott|
|     C00005|  carrie45@yahoo.com|   East Dennistown|   RI|         yahoo.com|  Sean Vasquez|
|     C00006|traceyramos@gmail...|     North Matthew|   IN|         gmail.com|Kevin Mccarthy|
|     C00007|scottallen@gmail.com|        Joneshaven|   VA|         gmail.com|  Amanda Doyle|
|     C00008|sullivanjeremy@ho...|  South Nathanfurt|   CT| 

In [9]:
df_wfn.write.format("delta")\
    .mode("overwrite")\
    .save("s3://learn-databricks-project-e2e-1-silver/customers")

In [10]:
%sql

CREATE SCHEMA IF NOT EXISTS learn_e2e_1.silver

""


In [13]:
%sql

CREATE TABLE IF NOT EXISTS learn_e2e_1.silver.customers
USING DELTA
LOCATION 's3://learn-databricks-project-e2e-1-silver/customers'

""
